# ETL


In [1]:
# session + warehouse schemas
import re
from pathlib import Path

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    DateType,
)

spark = (
    SparkSession.builder.appName("etl")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "spark-jupyter")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    # the worker has 2 cores total; without a cap one idle notebook session
    # holds them all and every later app waits forever for an executor
    .config("spark.cores.max", 2)
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)

print("app id :", spark.sparkContext.applicationId)
print("master :", spark.sparkContext.master)

RAW = "/opt/data/raw"
WAREHOUSE = "/opt/data/warehouse"

# scoped to the years that share one header style and one timestamp format:
# "Trip Id" / "Trip  Duration" (space-separated, BOM), dd/MM/yyyy HH:mm.
# 2021-2023 use the same header; 2024+ switch to underscores and add Bike_Model.
YEARS = [2019, 2020]
TIMESTAMP_FORMAT = "dd/MM/yyyy HH:mm"

TABLE_PATHS = {
    name: f"{WAREHOUSE}/{name}"
    for name in ("stage_trips", "dim_station", "dim_user", "fact_trips")
}

# stage_trips — every source column read as string; typing happens in transform
STAGE_TRIPS_COLUMNS = [
    "trip_id",
    "trip_duration",
    "start_station_id",
    "start_time",
    "start_station_name",
    "end_station_id",
    "end_time",
    "end_station_name",
    "bike_id",
    "user_type",
    "bike_model",  # absent in these years; kept null so the schema is stable
]

print("years  :", YEARS, "| warehouse:", WAREHOUSE)


app id : app-20260729031412-0008
master : spark://spark-master:7077
years  : [2019, 2020] | warehouse: /opt/data/warehouse


## Extract

Read the 2019–2020 CSVs, normalize the headers, add lineage columns, and write
`stage_trips` partitioned by `source_year`. Both years share one header style,
so the read asserts the schema rather than reconciling variants.


In [2]:
# extract — read raw csv, normalize headers, write stage_trips
def normalize(name: str) -> str:
    """lowercase, strip BOM, collapse whitespace/underscores to a single _"""
    return re.sub(r"[\s_]+", "_", name.replace("﻿", "").strip().lower())


def read_year(year: int):
    """Read one year of CSVs as all-string columns, normalized to STAGE_TRIPS_COLUMNS."""
    df = (
        spark.read.option("header", True)
        .option("encoding", "UTF-8")
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        # literal "NULL" appears in station names; treat it as missing
        .option("nullValue", "NULL")
        .csv(f"{RAW}/{year}/*.csv", schema=None, inferSchema=False)
    )

    df = df.toDF(*[normalize(c) for c in df.columns])

    # these years predate bike_model — fail loudly on any other schema drift
    unexpected = set(df.columns) - set(STAGE_TRIPS_COLUMNS)
    missing = set(STAGE_TRIPS_COLUMNS) - set(df.columns) - {"bike_model"}
    assert not unexpected, f"{year}: unexpected columns {sorted(unexpected)}"
    assert not missing, f"{year}: missing columns {sorted(missing)}"

    df = df.withColumn("bike_model", F.lit(None).cast(StringType()))

    return df.select(
        *[F.col(c).cast(StringType()).alias(c) for c in STAGE_TRIPS_COLUMNS],
        # input_file_name() gives the full path; keep "<year>/<file>.csv"
        F.regexp_extract(F.input_file_name(), r"([^/]+/[^/]+)$", 1).alias("source_file"),
        F.lit(year).cast(IntegerType()).alias("source_year"),
    )


stage_trips = read_year(YEARS[0])
for year in YEARS[1:]:
    stage_trips = stage_trips.unionByName(read_year(year))

(
    stage_trips.write.mode("overwrite")
    .partitionBy("source_year")
    .parquet(TABLE_PATHS["stage_trips"])
)

stage_trips = spark.read.parquet(TABLE_PATHS["stage_trips"])
raw_count = stage_trips.count()
print(f"years   : {YEARS}")
print(f"columns : {stage_trips.columns}")
print(f"rows    : {raw_count:,}")
stage_trips.groupBy("source_year").count().orderBy("source_year").show()


years   : [2019, 2020]
columns : ['trip_id', 'trip_duration', 'start_station_id', 'start_time', 'start_station_name', 'end_station_id', 'end_time', 'end_station_name', 'bike_id', 'user_type', 'bike_model', 'source_file', 'source_year']
rows    : 5,350,825
+-----------+-------+
|source_year|  count|
+-----------+-------+
|       2019|2439517|
|       2020|2911308|
+-----------+-------+



## Transform

Cast types, parse the two timestamp formats, drop invalid trips, deduplicate on
`trip_id`, then derive the calendar columns. Rejects are counted per rule.


In [3]:
# transform — type, parse, filter, deduplicate
rejects = {}


def drop_and_count(df, keep, rule: str, before: int) -> tuple:
    """Apply a keep-condition, record how many rows it removed."""
    out = df.where(keep).cache()
    after = out.count()
    rejects[rule] = before - after
    return out, after


# 1. type cast — non-numeric ids/duration become null and are dropped
typed = stage_trips.select(
    F.col("trip_id").cast(LongType()).alias("trip_id"),
    F.col("trip_duration").cast(IntegerType()).alias("trip_duration"),
    F.col("start_station_id").cast(IntegerType()).alias("start_station_id"),
    F.col("end_station_id").cast(IntegerType()).alias("end_station_id"),
    F.trim(F.col("start_station_name")).alias("start_station_name"),
    F.trim(F.col("end_station_name")).alias("end_station_name"),
    F.col("start_time").alias("start_time_raw"),
    F.col("end_time").alias("end_time_raw"),
    F.trim(F.col("user_type")).alias("user_type"),
)

typed, n = drop_and_count(
    typed, F.col("trip_id").isNotNull(), "trip_id not numeric", raw_count
)

# 2. parse timestamps — these years are all dd/MM/yyyy HH:mm
parsed = typed.withColumn(
    "start_time", F.to_timestamp("start_time_raw", TIMESTAMP_FORMAT)
).withColumn("end_time", F.to_timestamp("end_time_raw", TIMESTAMP_FORMAT))

parsed, n = drop_and_count(
    parsed,
    F.col("start_time").isNotNull() & F.col("end_time").isNotNull(),
    "timestamp unparseable",
    n,
)

# 3. filter invalid trips — duration recomputed from the parsed timestamps
parsed = parsed.withColumn(
    "trip_duration",
    (F.col("end_time").cast("long") - F.col("start_time").cast("long")).cast(
        IntegerType()
    ),
)

parsed, n = drop_and_count(parsed, F.col("end_time") > F.col("start_time"), "end <= start", n)
parsed, n = drop_and_count(
    parsed, F.col("trip_duration").between(60, 24 * 60 * 60), "duration out of range", n
)
parsed, n = drop_and_count(
    parsed,
    F.col("start_station_id").isNotNull() & F.col("end_station_id").isNotNull(),
    "station id null",
    n,
)

# 4. deduplicate — keep the first occurrence of each trip_id
deduped = parsed.dropDuplicates(["trip_id"]).cache()
clean_count = deduped.count()
rejects["duplicate trip_id"] = n - clean_count

print(f"stage rows : {raw_count:,}")
for rule, dropped in rejects.items():
    print(f"  rejected {dropped:>9,}  {rule}")
print(f"clean rows : {clean_count:,}  ({clean_count / raw_count:.1%} retained)")


stage rows : 5,350,825
  rejected         0  trip_id not numeric
  rejected 3,250,352  timestamp unparseable
  rejected     6,753  end <= start
  rejected    10,456  duration out of range
  rejected       281  station id null
  rejected         0  duplicate trip_id
clean rows : 2,082,983  (38.9% retained)


## Load

Build `dim_station` and `dim_user` first, then `fact_trips` with FK joins.


In [ ]:
# load — dim_station, dim_user
# every (id, name, time) sighting from both ends of the trip
sightings = deduped.select(
    F.col("start_station_id").alias("station_id"),
    F.col("start_station_name").alias("station_name"),
    F.col("start_time").alias("seen_at"),
).unionByName(
    deduped.select(
        F.col("end_station_id").alias("station_id"),
        F.col("end_station_name").alias("station_name"),
        F.col("end_time").alias("seen_at"),
    )
)

# where an id maps to several names across years, the latest sighting wins
latest_name = Window.partitionBy("station_id").orderBy(F.col("seen_at").desc())

dim_station = (
    sightings.where(F.col("station_id").isNotNull() & F.col("station_name").isNotNull())
    .withColumn("rank", F.row_number().over(latest_name))
    .where(F.col("rank") == 1)
    .select("station_id", "station_name")
)

dim_station.write.mode("overwrite").parquet(TABLE_PATHS["dim_station"])
dim_station = spark.read.parquet(TABLE_PATHS["dim_station"]).cache()

# dim_user — canonical labels; the surrogate key is fixed, not generated
dim_user = spark.createDataFrame(
    [(1, "member"), (2, "casual")], schema="user_type_id int, user_type string"
)
dim_user.write.mode("overwrite").parquet(TABLE_PATHS["dim_user"])
dim_user = spark.read.parquet(TABLE_PATHS["dim_user"]).cache()

print(f"dim_station : {dim_station.count():,} stations")
dim_station.orderBy("station_id").show(5, truncate=False)
print(f"dim_user    : {dim_user.count()} types")
dim_user.show()

# what the source actually contains, so unmapped labels are visible
deduped.groupBy("user_type").count().orderBy(F.col("count").desc()).show(truncate=False)


In [ ]:
# load — fact_trips
def calendar_columns(prefix: str, column):
    """year/month/date/day_of_week/hour/quarter derived from one timestamp."""
    return [
        F.year(column).cast(IntegerType()).alias(f"{prefix}_year"),
        F.month(column).cast(IntegerType()).alias(f"{prefix}_month"),
        F.to_date(column).alias(f"{prefix}_date"),
        # dayofweek() is 1=Sunday; the warehouse wants 1=Monday .. 7=Sunday
        ((F.dayofweek(column) + 5) % 7 + 1).cast(IntegerType()).alias(f"{prefix}_day_of_week"),
        F.hour(column).cast(IntegerType()).alias(f"{prefix}_hour"),
        F.quarter(column).cast(IntegerType()).alias(f"{prefix}_quarter"),
    ]


# conform user_type to the dim_user labels, then join for the surrogate key
conformed = deduped.withColumn(
    "user_type",
    F.when(F.lower("user_type").contains("annual"), "member")
    .when(F.lower("user_type") == "member", "member")
    .when(F.lower("user_type").contains("casual"), "casual"),
)

fact_trips = (
    conformed.join(F.broadcast(dim_user), on="user_type", how="left")
    .select(
        "trip_id",
        "trip_duration",
        *calendar_columns("start", F.col("start_time")),
        *calendar_columns("end", F.col("end_time")),
        "start_station_id",
        "end_station_id",
        "user_type_id",
    )
    # column order must match the warehouse schema
    .select(
        "trip_id",
        "trip_duration",
        "start_year",
        "start_month",
        "start_date",
        "start_day_of_week",
        "start_hour",
        "start_quarter",
        "end_year",
        "end_month",
        "end_date",
        "end_day_of_week",
        "end_hour",
        "end_quarter",
        "start_station_id",
        "end_station_id",
        "user_type_id",
    )
)

(
    fact_trips.write.mode("overwrite")
    .partitionBy("start_year", "start_month")
    .parquet(TABLE_PATHS["fact_trips"])
)

fact_trips = spark.read.parquet(TABLE_PATHS["fact_trips"])
fact_count = fact_trips.count()
print(f"fact_trips : {fact_count:,} rows")
fact_trips.orderBy("trip_id").show(5)


## Validation

Row counts raw → stage → fact with rejects itemized, no orphan FKs, and
`trip_id` unique in `fact_trips`.


In [ ]:
# validation
checks = []

# 1. row counts reconcile: stage - rejects = fact
print(f"stage_trips {raw_count:>12,}")
for rule, dropped in rejects.items():
    print(f"  -{dropped:>10,}  {rule}")
print(f"fact_trips  {fact_count:>12,}")
checks.append(("counts reconcile", raw_count - sum(rejects.values()) == fact_count))

# 2. no orphan FKs — every station id in the fact must exist in dim_station
station_ids = dim_station.select("station_id")
orphan_start = fact_trips.join(
    station_ids, fact_trips.start_station_id == station_ids.station_id, "left_anti"
).count()
orphan_end = fact_trips.join(
    station_ids, fact_trips.end_station_id == station_ids.station_id, "left_anti"
).count()
orphan_user = fact_trips.where(F.col("user_type_id").isNull()).count()

print(f"\norphan start_station_id : {orphan_start:,}")
print(f"orphan end_station_id   : {orphan_end:,}")
print(f"null user_type_id       : {orphan_user:,}")
checks.append(("no orphan station FKs", orphan_start == 0 and orphan_end == 0))
checks.append(("no null user_type_id", orphan_user == 0))

# 3. trip_id is unique
distinct_ids = fact_trips.select("trip_id").distinct().count()
print(f"\ndistinct trip_id : {distinct_ids:,} of {fact_count:,}")
checks.append(("trip_id unique", distinct_ids == fact_count))

print()
for name, passed in checks:
    print(f"  [{'PASS' if passed else 'FAIL'}] {name}")

fact_trips.groupBy("start_year").count().orderBy("start_year").show()
